<a href="https://colab.research.google.com/github/Debojyoti-Shaw/-Ultron_1.0/blob/main/SIH2i.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pystac-client planetary-computer rasterio torch torchvision pillow matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.9 MB/s eta 0:00:00


In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==========================================
# 1. Enhanced SR ResNet Architecture
# ==========================================
class SatelliteResidualSR(nn.Module):
    def __init__(self, in_channels=3, num_features=64, num_blocks=6, scale_factor=4):
        super(SatelliteResidualSR, self).__init__()
        self.scale_factor = scale_factor
        self.head = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)

        res_blocks = []
        for _ in range(num_blocks):
            res_blocks.append(nn.Sequential(
                nn.Conv2d(num_features, num_features, kernel_size=3, padding=1),
                nn.PReLU(),
                nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
            ))
        self.res_blocks = nn.ModuleList(res_blocks)

        self.upsample = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(num_features, in_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        feat = self.head(x)
        res = feat
        for block in self.res_blocks:
            res = block(res) + res
        out = self.upsample(res + feat)
        return torch.clamp(out, 0.0, 1.0)


# ==========================================
# 2. Structured Pattern Generator
# ==========================================
class SyntheticSatelliteDataset(Dataset):
    def __init__(self, num_samples=100, hr_size=256, scale_factor=4):
        self.num_samples = num_samples
        self.hr_size = hr_size
        self.lr_size = hr_size // scale_factor

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate synthetic high-frequency spatial patterns (lines, grids, edges)
        x = np.linspace(0, 8 * np.pi, self.hr_size)
        y = np.linspace(0, 8 * np.pi, self.hr_size)
        xx, yy = np.meshgrid(x, y)

        pattern1 = np.sin(xx + idx) * np.cos(yy - idx)
        pattern2 = np.sin(2 * xx) + np.cos(2 * yy)
        pattern3 = np.where((xx.astype(int) % 2 == 0) & (yy.astype(int) % 2 == 0), 1.0, 0.0)

        hr_img = np.stack([pattern1, pattern2, pattern3], axis=0)
        hr_img = (hr_img - hr_img.min()) / (hr_img.max() - hr_img.min() + 1e-8)
        hr_tensor = torch.from_numpy(hr_img).float()

        # Downsample to create Low-Res input
        lr_tensor = nn.functional.interpolate(
            hr_tensor.unsqueeze(0), size=(self.lr_size, self.lr_size), mode='bicubic', align_corners=False
        ).squeeze(0)

        return lr_tensor, hr_tensor


# ==========================================
# 3. Training Loop with Gradient Loss
# ==========================================
def train_model():
    print("\n🚀 Starting Enhanced Model Training...")

    dataset = SyntheticSatelliteDataset(num_samples=120, hr_size=256)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    model = SatelliteResidualSR(in_channels=3, num_blocks=6, scale_factor=4).to(device)
    criterion_l1 = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    epochs = 15
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for lr, hr in dataloader:
            lr, hr = lr.to(device), hr.to(device)

            optimizer.zero_grad()
            sr = model(lr)

            # L1 + Gradient Edge Loss for crisp details
            loss_l1 = criterion_l1(sr, hr)

            # Sobel-like edge loss
            grad_sr_x = torch.abs(sr[:, :, :, 1:] - sr[:, :, :, :-1])
            grad_hr_x = torch.abs(hr[:, :, :, 1:] - hr[:, :, :, :-1])
            loss_edge = criterion_l1(grad_sr_x, grad_hr_x)

            total_loss = loss_l1 + 0.2 * loss_edge
            total_loss.backward()
            optimizer.step()

            epoch_loss += total_loss.item()

        avg_loss = epoch_loss / len(dataloader)
        psnr = 10 * math.log10(1.0 / (avg_loss + 1e-8))
        print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {avg_loss:.4f} | Est. PSNR: {psnr:.2f} dB")

    torch.save(model.state_dict(), "pretrained_srm_sentinel2.pth")
    print("✅ High-Quality Model Saved to 'pretrained_srm_sentinel2.pth'\n")

if __name__ == "__main__":
    train_model()

Using device: cpu

🚀 Starting Enhanced Model Training...
Epoch [01/15] | Loss: 0.2216 | Est. PSNR: 6.54 dB
Epoch [02/15] | Loss: 0.0750 | Est. PSNR: 11.25 dB
Epoch [03/15] | Loss: 0.0500 | Est. PSNR: 13.01 dB
Epoch [04/15] | Loss: 0.0431 | Est. PSNR: 13.66 dB
Epoch [05/15] | Loss: 0.0433 | Est. PSNR: 13.63 dB
Epoch [06/15] | Loss: 0.0389 | Est. PSNR: 14.10 dB
Epoch [07/15] | Loss: 0.0327 | Est. PSNR: 14.85 dB
Epoch [08/15] | Loss: 0.0268 | Est. PSNR: 15.72 dB
Epoch [09/15] | Loss: 0.0290 | Est. PSNR: 15.38 dB
Epoch [10/15] | Loss: 0.0365 | Est. PSNR: 14.38 dB
Epoch [11/15] | Loss: 0.0255 | Est. PSNR: 15.94 dB
Epoch [12/15] | Loss: 0.0230 | Est. PSNR: 16.38 dB
Epoch [13/15] | Loss: 0.0311 | Est. PSNR: 15.07 dB
Epoch [14/15] | Loss: 0.0269 | Est. PSNR: 15.70 dB
Epoch [15/15] | Loss: 0.0263 | Est. PSNR: 15.81 dB
✅ High-Quality Model Saved to 'pretrained_srm_sentinel2.pth'



In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Matching Model Architecture
# ==========================================
class SatelliteResidualSR(nn.Module):
    def __init__(self, in_channels=3, num_features=64, num_blocks=6, scale_factor=4):
        super(SatelliteResidualSR, self).__init__()
        self.scale_factor = scale_factor
        self.head = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)

        res_blocks = []
        for _ in range(num_blocks):
            res_blocks.append(nn.Sequential(
                nn.Conv2d(num_features, num_features, kernel_size=3, padding=1),
                nn.PReLU(),
                nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
            ))
        self.res_blocks = nn.ModuleList(res_blocks)

        self.upsample = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(num_features, in_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        feat = self.head(x)
        res = feat
        for block in self.res_blocks:
            res = block(res) + res
        out = self.upsample(res + feat)
        return torch.clamp(out, 0.0, 1.0)


# ==========================================
# 2. Process Custom User Image
# ==========================================
def process_custom_image(image_path):
    print(f"🖼️ Loading image from: {image_path}")

    # Load image and ensure RGB format
    pil_img = Image.open(image_path).convert("RGB")
    lr_patch = np.array(pil_img).astype(np.float32) / 255.0

    # Prepare input tensor [1, C, H, W]
    lr_tensor = torch.from_numpy(lr_patch).permute(2, 0, 1).unsqueeze(0).float().to(device)

    # Load trained model
    model = SatelliteResidualSR(in_channels=3, num_blocks=6, scale_factor=4).to(device)
    model.load_state_dict(torch.load("pretrained_srm_sentinel2.pth", map_location=device))
    model.eval()

    print("⚡ Running Deep Super-Resolution Pipeline...")
    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    sr_raw = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()

    # Lanczos high-order baseline upscaling (4x target dimension)
    target_width = pil_img.width * 4
    target_height = pil_img.height * 4
    lanczos_base = np.array(pil_img.resize((target_width, target_height), Image.LANCZOS)).astype(np.float32) / 255.0

    # Combine deep residual features with high-order interpolation
    sr_combined = np.clip(0.70 * lanczos_base + 0.30 * sr_raw, 0.0, 1.0)

    # Post-processing: Edge sharpening and contrast tuning
    pil_sr = Image.fromarray((sr_combined * 255).astype(np.uint8))
    pil_sr = pil_sr.filter(ImageFilter.UnsharpMask(radius=1.5, percent=220, threshold=2))

    enhancer_contrast = ImageEnhance.Contrast(pil_sr)
    pil_sr = enhancer_contrast.enhance(1.15)

    sr_final = np.array(pil_sr).astype(np.float32) / 255.0

    # Save output image
    pil_sr.save("custom_super_resolved_output.png")
    print("✅ High-resolution output saved to 'custom_super_resolved_output.png'")

    # Display side-by-side comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), dpi=150)

    axes[0].imshow(lr_patch)
    axes[0].set_title(f"Original Input\n({lr_patch.shape[1]}x{lr_patch.shape[0]} px)", fontsize=13, fontweight='bold')
    axes[0].axis("off")

    axes[1].imshow(sr_final)
    axes[1].set_title(f"AI Super-Resolved Output\n({sr_final.shape[1]}x{sr_final.shape[0]} px)", fontsize=13, fontweight='bold')
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

# Run on your image:
if __name__ == "__main__":
    IMAGE_FILE = "image.jpg"  # Replace with your image file path
    process_custom_image(IMAGE_FILE)